# ScrapSense — Anomaly Model Analysis
Train, evaluate, and tune the Isolation Forest on synthetic reports.

In [2]:
import json, statistics
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import ParameterGrid

raw = json.loads(Path('synthethic_report.json').read_text(encoding='utf-8'))
df = pd.DataFrame(raw)
print(df.shape)
df.head()

(320, 9)


,material_type,quantity_kg,price_paid_pkr_per_kg,price_paid_pkr,location,timestamp,reported_by,is_synthetic,synthetic_label
0,glass,4.82,30.90,148.92,New Multan,2026-07-22T13:07:26.293450+00:00,picker_010,True,anomaly_price_too_high
1,plastic_hdpe,1.73,79.99,138.39,Cantt,2026-07-27T18:30:10.444899+00:00,picker_007,True,normal
2,other,0.78,56.34,43.95,Cantt,2026-07-23T02:04:27.409478+00:00,picker_009,True,normal
3,metal_steel,6.17,190.33,1174.35,Cantt,2026-07-19T01:05:23.177627+00:00,picker_009,True,normal
4,mixed_paper,6.16,35.80,220.50,Gulgasht Colony,2026-07-27T02:30:24.882466+00:00,picker_001,True,normal


## 1. Explore the data

In [3]:
print(df['synthetic_label'].value_counts())
print()
print(df['material_type'].value_counts())

synthetic_label
normal                     302
anomaly_price_too_high       7
anomaly_quantity_absurd      6
anomaly_price_too_low        5
Name: count, dtype: int64

material_type
glass             40
plastic_hdpe      40
other             40
metal_steel       40
mixed_paper       40
plastic_pet       40
metal_aluminum    40
cardboard         40
Name: count, dtype: int64


In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('All Reports by Material', 'Anomaly Detection'),
    horizontal_spacing=0.12
)

# --- Left panel: scatter grouped by material_type ---
colors_palette = ['#8FA8B2', '#C98A7D', '#A3B18A', '#B8A088', '#8E8FA3',
                   '#7A9E9F', '#C4A6A0', '#96A996', '#A89BB5', '#B0A488']

for i, (mat, grp) in enumerate(df.groupby('material_type')):
    fig.add_trace(
        go.Scatter(
            x=grp['quantity_kg'],
            y=grp['price_paid_pkr_per_kg'],
            mode='markers',
            name=mat,
            legendgroup=mat,
            marker=dict(size=6, opacity=0.65, color=colors_palette[i % len(colors_palette)],
                        line=dict(width=0.5, color='rgba(255,255,255,0.15)')),
            hovertemplate=(
                f'<b>{mat}</b><br>' +
                'Qty: %{x:.1f} kg<br>' +
                'Price: PKR %{y:.1f}/kg<extra></extra>'
            )
        ),
        row=1, col=1
    )

# --- Right panel: anomaly highlight ---
for label, color, name in [('normal', '#7A93A3', 'Normal'), ('anomaly', '#C97B6E', 'Anomaly')]:
    mask = df['synthetic_label'].eq('normal') if label == 'normal' else df['synthetic_label'].ne('normal')
    grp = df[mask]
    fig.add_trace(
        go.Scatter(
            x=grp['quantity_kg'],
            y=grp['price_paid_pkr_per_kg'],
            mode='markers',
            name=name,
            marker=dict(size=6, opacity=0.65, color=color,
                        line=dict(width=0.5, color='rgba(255,255,255,0.15)')),
            customdata=grp[['material_type', 'synthetic_label']],
            hovertemplate=(
                'Material: %{customdata[0]}<br>' +
                'Label: %{customdata[1]}<br>' +
                'Qty: %{x:.1f} kg<br>' +
                'Price: PKR %{y:.1f}/kg<extra></extra>'
            )
        ),
        row=1, col=2
    )

fig.update_layout(
    height=500, width=1100,
    title_text='ScrapSense — Price vs Quantity',
    hovermode='closest',
    template='plotly_dark',
    paper_bgcolor='#161616',
    plot_bgcolor='#1c1c1c',
    font=dict(color='#d8d8d8'),
    legend=dict(font=dict(size=9), groupclick='togglegroup',
                bgcolor='rgba(0,0,0,0)'),
    margin=dict(t=80)
)

fig.update_xaxes(title_text='Quantity (kg)', row=1, col=1, gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')
fig.update_xaxes(title_text='Quantity (kg)', row=1, col=2, gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')
fig.update_yaxes(title_text='Price (PKR/kg)', row=1, col=1, gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')
fig.update_yaxes(title_text='Price (PKR/kg)', row=1, col=2, gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')

fig.show()

## 2. Feature engineering (same as ml_anomaly_model.py)

In [5]:
# Per-material median price — normalises across materials
medians = df.groupby('material_type')['price_paid_pkr_per_kg'].median().to_dict()

df['price_ratio'] = df.apply(
    lambda r: r['price_paid_pkr_per_kg'] / medians[r['material_type']], axis=1)
df['log_qty'] = np.log1p(df['quantity_kg'])
df['is_anomaly'] = (df['synthetic_label'] != 'normal').astype(int)

X = df[['price_ratio', 'log_qty']].values
y = df['is_anomaly'].values
print(f"Features: {X.shape}  Anomaly rate: {y.mean():.2%}")

Features: (320, 2)  Anomaly rate: 5.62%


## 3. Baseline model (current settings)

In [7]:
baseline = IsolationForest(contamination=0.05, random_state=42)
baseline.fit(X)
preds = (baseline.predict(X) == -1).astype(int)  # -1 = anomaly

print(classification_report(y, preds, target_names=['normal', 'anomaly']))
print('ROC-AUC:', roc_auc_score(y, -baseline.score_samples(X)))

              precision    recall  f1-score   support

      normal       0.99      1.00      1.00       302
     anomaly       1.00      0.89      0.94        18

    accuracy                           0.99       320
   macro avg       1.00      0.94      0.97       320
weighted avg       0.99      0.99      0.99       320

ROC-AUC: 1.0


In [19]:
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, preds)
labels_x = ['pred_normal', 'pred_anomaly']
labels_y = ['true_normal', 'true_anomaly']

# Soft, low-saturation blue scale to match the muted palette
colorscale = [
    [0.0, '#1c1c1c'],
    [0.3, '#3a4a52'],
    [0.6, '#5f7d87'],
    [1.0, '#8FA8B2']
]

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels_x,
    y=labels_y,
    colorscale=colorscale,
    showscale=True,
    text=cm,
    texttemplate='%{text}',
    textfont=dict(size=16, color='#e8e8e8'),
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>',
    xgap=3,
    ygap=3
))

fig.update_layout(
    title_text='Baseline Confusion Matrix',
    height=450, width=500,
    template='plotly_dark',
    paper_bgcolor='#161616',
    plot_bgcolor='#1c1c1c',
    font=dict(color='#d8d8d8'),
    xaxis=dict(title='Predicted', side='bottom'),
    yaxis=dict(title='Actual', autorange='reversed'),
    margin=dict(t=60, l=80)
)

fig.show()

## 4. Hyperparameter tuning — contamination & n_estimators

In [9]:
grid = ParameterGrid({
    'contamination': [0.03, 0.05, 0.08, 0.10, 0.15],
    'n_estimators':  [50, 100, 200],
    'max_samples':   ['auto', 0.8],
})

results = []
for params in grid:
    m = IsolationForest(**params, random_state=42)
    m.fit(X)
    p = (m.predict(X) == -1).astype(int)
    auc = roc_auc_score(y, -m.score_samples(X))
    from sklearn.metrics import f1_score, precision_score, recall_score
    results.append({
        **params,
        'f1_anomaly':  f1_score(y, p, pos_label=1),
        'precision':   precision_score(y, p, pos_label=1, zero_division=0),
        'recall':      recall_score(y, p, pos_label=1),
        'roc_auc':     auc,
    })

results_df = pd.DataFrame(results).sort_values('f1_anomaly', ascending=False)
results_df.head(10)

,contamination,max_samples,n_estimators,f1_anomaly,precision,recall,roc_auc
11,0.05,0.8,200,0.941176,1.000000,0.888889,1.0
10,0.05,0.8,100,0.941176,1.000000,0.888889,1.0
9,0.05,0.8,50,0.941176,1.000000,0.888889,1.0
8,0.05,auto,200,0.941176,1.000000,0.888889,1.0
7,0.05,auto,100,0.941176,1.000000,0.888889,1.0
6,0.05,auto,50,0.941176,1.000000,0.888889,1.0
12,0.08,auto,50,0.818182,0.692308,1.000000,1.0
13,0.08,auto,100,0.818182,0.692308,1.000000,1.0
15,0.08,0.8,50,0.818182,0.692308,1.000000,1.0
14,0.08,auto,200,0.818182,0.692308,1.000000,1.0


## 5. Best model — detailed evaluation

In [10]:
best = results_df.iloc[0]
print('Best params:', best[['contamination','n_estimators','max_samples']].to_dict())

best_model = IsolationForest(
    contamination=best['contamination'],
    n_estimators=int(best['n_estimators']),
    max_samples=best['max_samples'],
    random_state=42,
)
best_model.fit(X)
best_preds = (best_model.predict(X) == -1).astype(int)
print(classification_report(y, best_preds, target_names=['normal', 'anomaly']))

Best params: {'contamination': 0.05, 'n_estimators': 200, 'max_samples': 0.8}
              precision    recall  f1-score   support

      normal       0.99      1.00      1.00       302
     anomaly       1.00      0.89      0.94        18

    accuracy                           0.99       320
   macro avg       1.00      0.94      0.97       320
weighted avg       0.99      0.99      0.99       320



In [20]:
import plotly.graph_objects as go

scores = best_model.score_samples(X)

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=scores[y == 0],
    nbinsx=40,
    name='normal',
    marker=dict(color='#7A93A3', line=dict(width=0.5, color='rgba(255,255,255,0.15)')),
    opacity=0.65,
    hovertemplate='Score: %{x:.3f}<br>Count: %{y}<extra>normal</extra>'
))

fig.add_trace(go.Histogram(
    x=scores[y == 1],
    nbinsx=40,
    name='anomaly',
    marker=dict(color='#C97B6E', line=dict(width=0.5, color='rgba(255,255,255,0.15)')),
    opacity=0.65,
    hovertemplate='Score: %{x:.3f}<br>Count: %{y}<extra>anomaly</extra>'
))

fig.update_layout(
    barmode='overlay',
    title_text='Score Distribution by True Label',
    xaxis_title='Anomaly score (lower = more anomalous)',
    yaxis_title='Count',
    height=450, width=900,
    template='plotly_dark',
    paper_bgcolor='#161616',
    plot_bgcolor='#1c1c1c',
    font=dict(color='#d8d8d8'),
    legend=dict(font=dict(size=10), bgcolor='rgba(0,0,0,0)'),
    bargap=0.02
)

fig.update_xaxes(gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')
fig.update_yaxes(gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')

fig.show()

In [23]:
import numpy as np
import plotly.graph_objects as go

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.2, X[:,0].max()+0.2, 300),
                     np.linspace(X[:,1].min()-0.2, X[:,1].max()+0.2, 300))
Z = best_model.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Muted diverging colorscale (low score = warm/red-ish, high score = cool/green-ish)
colorscale = [
    [0.0, '#B0685E'],   # muted red — most anomalous
    [0.5, '#8E8A6D'],   # muted olive — neutral
    [1.0, '#6E8E6E']    # muted green — most normal
]

fig = go.Figure()

# Decision surface
fig.add_trace(go.Contour(
    x=xx[0],
    y=yy[:, 0],
    z=Z,
    colorscale=colorscale,
    opacity=0.65,
    contours=dict(showlines=False),
    colorbar=dict(title='anomaly<br>score', tickfont=dict(size=9)),
    hoverinfo='skip'
))

# Normal points
fig.add_trace(go.Scatter(
    x=X[y==0, 0], y=X[y==0, 1],
    mode='markers',
    name='normal',
    marker=dict(size=6, color='#7A93A3', opacity=0.75,
                line=dict(width=0.5, color='rgba(255,255,255,0.2)')),
    hovertemplate='price_ratio: %{x:.3f}<br>log(1+qty): %{y:.3f}<extra>normal</extra>'
))

# Anomaly points
fig.add_trace(go.Scatter(
    x=X[y==1, 0], y=X[y==1, 1],
    mode='markers',
    name='anomaly',
    marker=dict(size=9, color='#E8E8E8', symbol='x',
                line=dict(width=1.5, color='#C97B6E')),
    hovertemplate='price_ratio: %{x:.3f}<br>log(1+qty): %{y:.3f}<extra>anomaly</extra>'
))

fig.update_layout(
    title_text='Isolation Forest Decision Surface',
    xaxis_title='price_ratio',
    yaxis_title='log(1+quantity_kg)',
    height=600, width=850,
    template='plotly_dark',
    paper_bgcolor='#161616',
    plot_bgcolor='#1c1c1c',
    font=dict(color='#d8d8d8'),
    legend=dict(font=dict(size=10), bgcolor='rgba(0,0,0,0)')
)

fig.update_xaxes(gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')
fig.update_yaxes(gridcolor='#2a2a2a', zerolinecolor='#3a3a3a')

fig.show()

## 6. Per-material breakdown

In [14]:
from sklearn.metrics import f1_score
df['pred_anomaly'] = best_preds

rows = []
for mat, grp in df.groupby('material_type'):
    rows.append({
        'material': mat,
        'n': len(grp),
        'true_anomalies': grp['is_anomaly'].sum(),
        'pred_anomalies': grp['pred_anomaly'].sum(),
        'f1': f1_score(grp['is_anomaly'], grp['pred_anomaly'], pos_label=1, zero_division=0),
    })
pd.DataFrame(rows)

,material,n,true_anomalies,pred_anomalies,f1
0,cardboard,40,2,2,1.0
1,glass,40,2,2,1.0
2,metal_aluminum,40,1,1,1.0
3,metal_steel,40,2,2,1.0
4,mixed_paper,40,3,2,0.8
5,other,40,3,2,0.8
6,plastic_hdpe,40,4,4,1.0
7,plastic_pet,40,1,1,1.0


## 7. Save best model

In [15]:
import pickle
Path('models').mkdir(exist_ok=True)
with open('models/isolation_forest.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Saved models/isolation_forest.pkl')
print(f"contamination={best['contamination']}  n_estimators={int(best['n_estimators'])}  "
      f"f1={best['f1_anomaly']:.3f}  roc_auc={best['roc_auc']:.3f}")

Saved models/isolation_forest.pkl
contamination=0.05  n_estimators=200  f1=0.941  roc_auc=1.000
